In [1]:
# 학습된 모델 대신 직접 생성한 embedding을 사용(toy-embedding)
import math
import re
import pandas as pd
import torch
import torch.nn.functional as F

In [2]:
sentences = [
    '이 영화 정말 좋다',
    '이 영화 너무 지루하다',
    '배우 연기가 감동적이다',
    '전개는 보통이지만 마지막이 재미있다'
]

In [3]:
# toy-embedding : 2차원 벡터를 임의로 생성
# positive단어는 오른쪽 위 쪽에 배치
# negative 단어는 왼쪽 아래쪽에 배치
# neutal 단어는 원점 근처에 배치

# query와 비슷한 방향의 단어가 더 큰 score를 받음
toy_vocab = {
    '이': torch.tensor([0.0, 0.0]),
    '영화': torch.tensor([0.1, 0.0]),
    '정말': torch.tensor([0.0, 0.2]),
    '좋다': torch.tensor([1.2, 1.0]),
    '너무': torch.tensor([0.0, 0.1]),
    '지루하다': torch.tensor([-1.2, -1.0]),
    '배우': torch.tensor([0.2, 0.1]),
    '연기가': torch.tensor([0.3, 0.1]),
    '감동적이다': torch.tensor([1.0, 1.2]),
    '전개는': torch.tensor([0.0, 0.0]),
    '보통이지만': torch.tensor([0.0, 0.0]),
    '마지막이': torch.tensor([0.0, 0.1]),
    '재미있다': torch.tensor([1.1, 1.1]),
    '별로다': torch.tensor([-1.0, -1.1]),
    '최악이다': torch.tensor([-1.3, -1.2]),
}

positive_query = torch.tensor([1.0,1.0])
negative_query = torch.tensor([-1.0,-1.0])
toy_vocab['좋다'], positive_query, negative_query

(tensor([1.2000, 1.0000]), tensor([1., 1.]), tensor([-1., -1.]))

## 실제 Word2Vec Embedding 학습

In [4]:
from gensim.models import Word2Vec
import pandas as pd
from preprocessing import preprocess_dataframe,build_vocab

DEFAULT_URL = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df =pd.read_csv(DEFAULT_URL)

# 감정레이블 생성
df['sentiment'] = (df['rating']>=6).astype(int)
train_df = df[['review','sentiment']]
de_processed, vocab = preprocess_dataframe(train_df, 
                                           text_col='review', label_col='sentiment',max_len=30,min_freq=2)
word_to_idx = vocab
idx_to_word = { v:k for k,v in vocab.items()}
sentences = []
for tokens in de_processed['token_ids']:
    if len(tokens) > 0:
        words = [idx_to_word.get(idx, '<UNK>') for idx in tokens  if idx in idx_to_word]
        sentences.append(words)

In [ ]:
# word2vec 학습  전이학습
w2v_model =  Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=5,
    seed=42
)
print(f'word2vec 모델 학습 완료 :vector size = {w2v_model.vector_size}')

word2vec 모델 학습 완료 :vector size = 100


## toy_vocab을 Word2Vec embedding으로 업데이트....

In [15]:
toy_vocab = {}
for word in vocab:
    if word in w2v_model.wv:
        toy_vocab[word] = torch.from_numpy(w2v_model.wv[word]).float()
    else:
        toy_vocab[word] = torch.zeros(w2v_model.vector_size).float()
print(f'{len(toy_vocab)}개 단어의 embedding 준비')
print(f'embedding 차원 : {w2v_model.vector_size}')
sample_word = ['좋다','지루하다','재미있다']
for word in sample_word:
    if word in toy_vocab:
        vec = toy_vocab[word]
        print(f'{word} : {vec[:5]} ...')

# query를 100차원으로 업데이트
positive_query = torch.ones(w2v_model.vector_size)*0.1
negative_query = torch.ones(w2v_model.vector_size)* -0.1
print(f'positive_query size : {positive_query.shape}')
print(f'negative_query size : {negative_query.shape}')

14928개 단어의 embedding 준비
embedding 차원 : 100
좋다 : tensor([-0.0121,  0.1138,  0.1624, -0.1536,  0.0083]) ...
지루하다 : tensor([ 0.0038,  0.0409,  0.0581, -0.0404, -0.0029]) ...
재미있다 : tensor([ 0.0148,  0.0642,  0.0860, -0.0757, -0.0096]) ...
positive_query size : torch.Size([100])
negative_query size : torch.Size([100])


## 전처리 함수 만들기
- 문장을 토큰으로나눈뒤, 각 토큰을 toy embedding 벡터로 바꾼다
- 여기서 key value는 같은 벡터를 사용, query는 지금 찾고싶은 의미

In [16]:
def simple_tokenize(text: str):
    text = re.sub(r'[^가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    return text.split()

def get_token_vectors(tokens, vocab):
    vectors = []
    for token in tokens:
        vectors.append(vocab.get(token, torch.tensor([0.0, 0.0])))
    return torch.stack(vectors) if vectors else torch.empty(0, 2)

def attention_with_query(tokens, query, vocab):
    # ===== Step 1: 각 토큰을 벡터로 바꾼다 =====
    # tokens를 toy_vocab에서 찾아서 벡터로 변환한다.
    # 예: '좋다' -> [1.2, 1.0]
    key = get_token_vectors(tokens, vocab)
    
    # 간단한 예제에서는 key와 value를 같게 둔다.
    # (실제 모델에서는 key와 value를 만드는 다른 layer가 있다)
    value = key
    
    # 토큰이 없으면 빈 결과를 반환한다
    if key.numel() == 0:
        return pd.DataFrame(), torch.tensor([])
    
    # ===== Step 2: query와 key의 유사도를 계산한다 (score) =====
    # score = query · key / sqrt(d_k)
    # dot product는 두 벡터가 같은 방향일수록 크다.
    # 예: positive_query [1.0, 1.0]와 '좋다' [1.2, 1.0]의 dot product는 크다
    # 예: positive_query [1.0, 1.0]와 '지루하다' [-1.2, -1.0]의 dot product는 작다
    d_k = key.size(-1)  # d_k = 2 (toy embedding 차원)
    scores = torch.matmul(key, query) / math.sqrt(d_k)
    # scores shape: [토큰 개수]
    # 예: scores = [0.0, 0.05, 0.1, 1.55] 같은 값들
    
    # ===== Step 3: score를 확률처럼 바꾼다 (weight) =====
    # softmax를 사용해서 score들을 0~1 사이의 값으로 정규화한다.
    # 모든 weight의 합은 1이 된다.
    # score가 높을수록 weight도 크다.
    # 예: scores = [0.0, 0.1, 0.1, 1.5] -> weights = [0.1, 0.14, 0.14, 0.6]
    weights = F.softmax(scores, dim=0)
    # weights shape: [토큰 개수]
    
    # ===== Step 4: weight를 이용해서 value들을 섞는다 (context) =====
    # context = sum(weight_i * value_i)
    # 이것이 attention의 핵심이다.
    # weight가 높은 토큰의 value가 더 많이 더해진다.
    # 예: 좋다의 weight가 0.6이면, 좋다 벡터 [1.2, 1.0]이 60% 반영된다
    context = torch.sum(weights.unsqueeze(-1) * value, dim=0)
    # context shape: [2] (toy embedding 차원)

    frame = pd.DataFrame({
        'token': tokens,
        'key_x': key[:, 0].tolist(),
        'key_y': key[:, 1].tolist(),
        'score': scores.tolist(),
        'weight': weights.tolist(),
    })
    frame['value_x'] = frame['key_x']
    frame['value_y'] = frame['key_y']
    return frame, context

## 한문장에 query/ key /value 
- 첫번째 문장에 positive query를 넣어서 어떤 토큰이 더 중요한지 확인

In [17]:
toy_vocab

{'<pad>': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.]),
 '<unk>': tensor([-0.5507,  0.3725,  0.5184, -0.5763,  0.1665,  1.1814, -1.0070,  0.5792,
         -0.7467, -1.5329, -0.0371,  1.1699, -0.3358,  0.5901,  0.4417, -0.1705,
         -0.6230, -1.6911,  0.2608, -0.8807,  0.1473,  0.9436,  0.1735,  2.0403,
          0.5331,  1.3262, -1.6798,  0.3820, -0.4491, -0.7260, -0.4196, -0.7738,
         -0.5360,  0.3963,  0.9627, -0.2949,  0.6457, -0.0573, -1.4101,  1.0027,
          0.6405, -0.3148,  0.8359, -0.8370, -0.6879,  1.2232, -0.4390,  1.5968,
          0.8323,  0.6319,  0.4080, -0.9994, -0.4

In [18]:
sentence = sentences[0]
tokens = simple_tokenize(sentence)
frame,context = attention_with_query(tokens,positive_query,toy_vocab)
print('sentence:', sentence)
print('tokens:', tokens)
print('query:',positive_query.tolist())
print('context:',context.tolist())
display(frame.sort_values('weight', ascending=False).reset_index(drop=True))

TypeError: expected string or bytes-like object

- 의미해석 : 각 토큰을 toy-vector로 바뀐뒤 query와 의 유사도를 계산
- weight 는 score에 softmax를 적용한 확률기반의 점수
- 모델은 이문장에서 좋다 라는 단어를 중요하게 본다
- context: [0.7274695634841919, 0.6239237189292908]는 이 가중치를 반영해 만든 문장요약 벡터
- 결국 attention은 문장 전테를 다 똑같이 보지 않고 지금 query와 가장 관련있는 단어에 집중한다 

```
query = 지금 찾고 싶은 감정 방향
key = 각 단어가 가진 특징
score = query와 key가 얼마나 비슷한가
weight = score를 확률처럼 바꾼 값
context = weight로 단어들을 섞어 만든 최종 요약
```

## 같은 문장을 positive query와 negative query로 비교

query가 바뀌면 attention weight도 바뀐다.

- positive query: 좋은 단어 쪽을 더 본다.
- negative query: 나쁜 단어 쪽을 더 본다.

In [37]:
compare_sentence = '이 영화 정말 좋다 하지만 마지막이 별로다'
tokens = simple_tokenize(compare_sentence)
pos_trame, pos_context = attention_with_query(tokens,positive_query,toy_vocab)
neg_trame, neg_context = attention_with_query(tokens,negative_query,toy_vocab)

print('sentence:', sentence)
print('tokens:', tokens)
print('query:',positive_query.tolist())
print('context:',pos_context.tolist())
display(pos_trame.sort_values('weight', ascending=False).reset_index(drop=True))

print('sentence:', sentence)
print('tokens:', tokens)
print('query:',negative_query.tolist())
print('context:',neg_context.tolist())
display(neg_trame.sort_values('weight', ascending=False).reset_index(drop=True))

sentence: 이 영화 정말 좋다
tokens: ['이', '영화', '정말', '좋다', '하지만', '마지막이', '별로다']
query: [1.0, 1.0]
context: [0.5423838496208191, 0.47029146552085876]


,token,key_x,key_y,score,weight,value_x,value_y
0,좋다,1.2,1.0,1.555635,0.461665,1.2,1.0
1,정말,0.0,0.2,0.141421,0.112238,0.0,0.2
2,영화,0.1,0.0,0.070711,0.104576,0.1,0.0
3,마지막이,0.0,0.1,0.070711,0.104576,0.0,0.1
4,이,0.0,0.0,0.000000,0.097437,0.0,0.0
5,하지만,0.0,0.0,0.000000,0.097437,0.0,0.0
6,별로다,-1.0,-1.1,-1.484924,0.022071,-1.0,-1.1


sentence: 이 영화 정말 좋다
tokens: ['이', '영화', '정말', '좋다', '하지만', '마지막이', '별로다']
query: [-1.0, -1.0]
context: [-0.43476250767707825, -0.46789708733558655]


,token,key_x,key_y,score,weight,value_x,value_y
0,별로다,-1.0,-1.1,1.484924,0.471786,-1.0,-1.1
1,하지만,0.0,0.0,0.000000,0.106869,0.0,0.0
2,이,0.0,0.0,0.000000,0.106869,0.0,0.0
3,마지막이,0.0,0.1,-0.070711,0.099573,0.0,0.1
4,영화,0.1,0.0,-0.070711,0.099573,0.1,0.0
5,정말,0.0,0.2,-0.141421,0.092775,0.0,0.2
6,좋다,1.2,1.0,-1.555635,0.022555,1.2,1.0


## 여러 문장 한 번에 비교

문장마다 어떤 단어에 attention이 많이 가는지 표로 비교한다.

In [39]:
rows = []
for sentence in sentences:
    tokens = simple_tokenize(sentence)
    frame, context = attention_with_query(tokens, negative_query, toy_vocab)
    if len(frame) == 0:
        rows.append({
            'sentence': sentence,
            'top_token': None,
            'top_weight': None,
            'context_x': None,
            'context_y': None,
        })
        continue

    top_row = frame.sort_values('weight', ascending=False).iloc[0]
    rows.append({
        'sentence': sentence,
        'top_token': top_row['token'],
        'top_weight': round(float(top_row['weight']), 4),
        'context_x': round(float(context[0]), 4),
        'context_y': round(float(context[1]), 4),
    })

summary = pd.DataFrame(rows)
summary

,sentence,top_token,top_weight,context_x,context_y
0,이 영화 정말 좋다,이,0.3321,0.1151,0.1278
1,이 영화 너무 지루하다,지루하다,0.6233,-0.7357,-0.6110
2,배우 연기가 감동적이다,배우,0.4561,0.3377,0.2309
3,전개는 보통이지만 마지막이 재미있다,전개는,0.3182,0.0739,0.1035
